<a href="https://colab.research.google.com/github/Cibrium/2026_Intro_Python/blob/main/FEEDBACK_CiaranKeegan_1_2_data_structures_and_control_flow_environmental_exercise_1_step_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the previous step in the question containg the data for step three where I will demonstrate my error, cause and solution.


In [1]:
import pooch

data_file = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/station_iib_daily_max_temp_2022.csv",
    known_hash="sha256:034755fb289b4e157e5d029995481786a359eb25a80df62c09ee83d063013144",
    fname="station_iib_daily_max_temp_2022.csv",
    path=pooch.os_cache("mlees"),
)
print("data cached at:", data_file)

from pathlib import Path

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines    = fhandler.readlines()
day_count    =  0
temp_skipped =  0

for line in lines[1:]:               #was [1:] which showed all, [1:4] shows first 3
    fields = line.strip().split(",")
    #print(fields[1:3])

    month = fields[1].split(".")
    temperature = fields[2]

    if temperature == "" :
      temp_skipped += 1
      continue


    temperature = float(temperature)
    day_count += 1


total = temp_skipped + day_count

print(total)
print(temp_skipped)
print(day_count)



data cached at: /root/.cache/mlees/station_iib_daily_max_temp_2022.csv
365
16
349


Step 3. Build a dictionary mapping each month to the number of observations in it.

You do not know in advance which months are present, so the dictionary has to grow as you go. Keep skipping the days with no reading: a day the sensor did not record is not an observation, so October and November should come out short of their full length.

In [19]:
from pathlib import Path

counts = {"month": "number of observations"}

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines = fhandler.readlines()

for line in lines[1:]:
    fields = line.strip().split(",")

    month = fields[1].split(".")
    temperature = fields[2]

    if temperature == "":
        continue

    counts[month] = counts.get(month, 0) + 1

print(counts)

TypeError: unhashable type: 'list'

**Error number 1:** I have created a dictionary named "counts", assinging the key to the word "month" and the value to the "number of observations"

**count = {"month": "number of observatoins"}
counts.get(month,0)**

This was supposed to create a dictionary which would store the number of observation per month, what I actually created was a dictionary containing the literal key labled "month" and a value that was a string. This dictionary was not able to grow for values in the real data.

When I ran the code I received TypeError: unhashable type
: "list". This happened because the variable "month" was a list and not a value, lists cannot be used as a key in python.

To fix this error I had to create an empty dictionary so that I could build on the values from each month and grow the dictionary I did this by

**count = {}**

Now with this empty dictionary I could build using the data set. However when I ran the code again I recieved the same error code again.

In [22]:
from pathlib import Path

counts = {}

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines = fhandler.readlines()

for line in lines[1:]:
    fields = line.strip().split(",")

    month = fields[1].split(".")
    temperature = fields[2]

    if temperature == "":
        continue

    counts[month] = counts.get(month, 0) + 1

print(counts)


TypeError: unhashable type: 'list'

**Error number 2** after changing the dictionary from a static or non growable dictionary to a now empty dictionary count = {}, I ran the code again to find again, I had the same error TypeError: unhashable type: 'list'.

This indicated that there was still something wrong with my list so I used checked the value month, and checked its type




In [24]:
print(month)
print(type(month))

['01', '01', '22']
<class 'list'>


The result above shows that the month key is returning a list of three strings containing the day, month and year. Not a single value for each month. The issue was coming from how I separated the dates. I separated the dates everytime there was a full stop using split(".")

**month = fields[1].split(".")**

this set the month to a list of value

**['01', '01', '22']**

In order to fix this I had to add a selection square bracket after the split.

**month = fields[1].split(".")[1]**

now the dictionary key was selected as the second position which was equivalent to the actual month.


In [28]:
from pathlib import Path

counts = {}

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines = fhandler.readlines()

for line in lines[1:]:
    fields = line.strip().split(",")

    month = fields[1].split(".")[1]
    temperature = fields[2]

    if temperature == "":
        continue

    counts[month] = counts.get(month, 0) + 1

print(counts)

{'01': 31, '02': 28, '03': 31, '04': 30, '05': 31, '06': 30, '07': 31, '08': 31, '09': 30, '10': 24, '11': 21, '12': 31}


| Step | Issue Identified | Diagnostic Action | Resolution |
| :--- | :--- | :--- | :--- |
| **1. Dictionary Setup** | Stored a non functional value as key | Code review | Initialized empty dictionary `counts = {}` |
| **2. Key Selection** | Passed list `['01', '01', '22']` as dictionary key $\rightarrow$ `TypeError` | Inspected variable type with `type(month)` | Indexed month segment directly: `fields[1].split(".")[1]` |